In [ ]:
!pip install hijri_converter
"""
Ash Wednesday vs Ramadan — Enhanced Visualizations & Analysis
=============================================================
Hybrid dating: Umm al-Qura (1924–2077) + linear astronomical approximation outside.
Range: 1500–2099 (600 lat).
"""

import warnings
warnings.filterwarnings('ignore')

from datetime import date, timedelta
from hijri_converter import Hijri, Gregorian, ummalqura
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
from collections import defaultdict
import os

# ── Output dir ────────────────────────────────────────────────────────
OUT = '/home/claude/output'
os.makedirs(OUT, exist_ok=True)

# ── Matplotlib dark theme ─────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': '#0e1117',
    'axes.facecolor': '#0e1117',
    'text.color': '#e0e0e0',
    'axes.labelcolor': '#c0c0c0',
    'xtick.color': '#a0a0a0',
    'ytick.color': '#a0a0a0',
    'grid.color': '#2a2a3a',
    'grid.alpha': 0.5,
})

# ── Palette ───────────────────────────────────────────────────────────
C_LENT   = '#f97316'
C_RAM    = '#06b6d4'
C_ACCENT = '#facc15'
C_BG     = '#0e1117'
C_CARD   = '#161b22'
C_GRID   = '#2a2a3a'
C_TEXT   = '#e0e0e0'
C_MUTED  = '#6b7280'

MONTH_PL = ['Sty','Lut','Mar','Kwi','Maj','Cze','Lip','Sie','Wrz','Paź','Lis','Gru']
MONTH_DOY = [date(2000,m,1).timetuple().tm_yday for m in range(1,13)]

# ══════════════════════════════════════════════════════════════════════
#  DATE CALCULATIONS
# ══════════════════════════════════════════════════════════════════════

def easter_sunday(year: int) -> date:
    """Meeus/Jones/Butcher — Gregorian Easter."""
    a = year % 19
    b, c = divmod(year, 100)
    d, e = divmod(b, 4)
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = divmod(c, 4)
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month, day = divmod(h + l - 7 * m + 114, 31)
    return date(year, month, day + 1)

def ash_wednesday(year: int) -> date:
    return easter_sunday(year) - timedelta(days=46)

# ── Hybrid Ramadan calculator ─────────────────────────────────────────
(UQ_MIN_Y, _, _), (UQ_MAX_Y, _, _) = ummalqura.GREGORIAN_RANGE

# Anchor for linear extrapolation (from Umm al-Qura)
_ANCHOR_HY = 1446
_ANCHOR_GREG = date(2025, 3, 1)   # 1 Ramadan 1446 H (Umm al-Qura)
_ISLAMIC_YEAR = 354.36667          # mean length of Islamic year (30-year cycle)

def _ramadan_approx_linear(hijri_year: int) -> date:
    """1 Ramadan by linear extrapolation from anchor. Error ≤ ±2 days."""
    delta = round((hijri_year - _ANCHOR_HY) * _ISLAMIC_YEAR)
    return _ANCHOR_GREG + timedelta(days=delta)

def _hijri_year_for_greg(year: int) -> int:
    """Approximate Hijri year for Jan 1 of given Gregorian year."""
    # Rough formula: HY ≈ (GY - 622) * 365.25 / 354.367 + 1
    return int((year - 622) * 365.2425 / _ISLAMIC_YEAR) + 1

def _try_ummalqura(hijri_year: int):
    """Try Umm al-Qura; return date or None."""
    try:
        g = Hijri(hijri_year, 9, 1).to_gregorian()
        d = date(g.year, g.month, g.day)
        if UQ_MIN_Y <= d.year <= UQ_MAX_Y:
            return d
    except (ValueError, OverflowError):
        pass
    return None

def ramadan_starts_in_year(year: int) -> list[date]:
    """Return all 1-Ramadan dates falling in given Gregorian year.
    Uses Umm al-Qura when available, linear approximation otherwise."""
    results = []
    hy_approx = _hijri_year_for_greg(year)

    for hy in range(hy_approx - 1, hy_approx + 2):
        if hy < 1:
            continue
        # Try Umm al-Qura first
        d = _try_ummalqura(hy)
        if d is None:
            d = _ramadan_approx_linear(hy)
        if d.year == year:
            results.append(d)

    return sorted(set(results))

# ══════════════════════════════════════════════════════════════════════
#  GENERATE DATA
# ══════════════════════════════════════════════════════════════════════

START_YEAR = 1800
END_YEAR   = 2200

rows = []
for y in range(START_YEAR, END_YEAR + 1):
    aw = ash_wednesday(y)
    for rd in ramadan_starts_in_year(y):
        diff = (rd - aw).days
        rows.append((y, aw, rd, diff, diff == 0))

print(f"Wygenerowano {len(rows)} par dat ({START_YEAR}–{END_YEAR})")

# ── Derived arrays ────────────────────────────────────────────────────
years   = np.array([r[0] for r in rows])
aw_doy  = np.array([r[1].timetuple().tm_yday for r in rows])
rd_doy  = np.array([r[2].timetuple().tm_yday for r in rows])
diffs   = np.array([r[3] for r in rows])
sames   = np.array([r[4] for r in rows])
abs_diffs = np.abs(diffs)

same_years = sorted(years[sames])
print(f"Pokrycia (Δ=0): {same_years}")

# Flag which rows used UaQ vs approximation
is_uaq = np.array([UQ_MIN_Y <= r[0] <= UQ_MAX_Y for r in rows])

# ══════════════════════════════════════════════════════════════════════
#  CHART 1 — HERO TIMELINE
# ══════════════════════════════════════════════════════════════════════

def chart_hero():
    fig, axes = plt.subplots(2, 1, figsize=(20, 12),
                             gridspec_kw={'height_ratios': [3, 1.2]},
                             facecolor=C_BG)
    ax1, ax2 = axes

    # Shading for UaQ vs approximation zone
    ax1.axvspan(UQ_MIN_Y, UQ_MAX_Y, alpha=0.04, color='white', zorder=0)
    ax2.axvspan(UQ_MIN_Y, UQ_MAX_Y, alpha=0.04, color='white', zorder=0)

    # Top: Day-of-year
    ax1.fill_between(years, aw_doy, rd_doy, alpha=0.05, color='white')
    ax1.plot(years, aw_doy, '-', color=C_LENT, linewidth=1.2, alpha=0.85,
             label='Środa Popielcowa')
    ax1.plot(years, rd_doy, '-', color=C_RAM, linewidth=1.2, alpha=0.85,
             label='1. Ramadan')

    for i, s in enumerate(sames):
        if s:
            ax1.plot(years[i], aw_doy[i], 'o', color=C_ACCENT, markersize=11,
                     zorder=5, markeredgecolor='white', markeredgewidth=1.5)
            ax1.annotate(str(years[i]),
                         (years[i], aw_doy[i]),
                         fontsize=8, fontweight='bold', color=C_ACCENT,
                         textcoords='offset points', xytext=(0, 14), ha='center',
                         arrowprops=dict(arrowstyle='-', color=C_ACCENT, lw=0.7))

    ax1.set_yticks(MONTH_DOY)
    ax1.set_yticklabels(MONTH_PL, fontsize=9)
    ax1.set_ylim(1, 366)
    ax1.invert_yaxis()
    ax1.set_title('Wielki Post i Ramadan — kiedy się zaczynają? (1800–2200)',
                  fontsize=17, fontweight='bold', color='white', pad=14)
    ax1.set_ylabel('Miesiąc w roku', fontsize=11)
    ax1.legend(fontsize=11, loc='upper right',
               facecolor=C_CARD, edgecolor=C_GRID, labelcolor=C_TEXT)
    ax1.grid(True, axis='y', linewidth=0.4)

    # Data quality note
    ax1.text(UQ_MIN_Y + (UQ_MAX_Y - UQ_MIN_Y) / 2, 355,
             'Umm al-Qura (±0 dni)', fontsize=8, color=C_MUTED,
             ha='center', va='top', alpha=0.7)
    ax1.text(START_YEAR + 30, 355,
             'aproksymacja (±1–2 dni)', fontsize=8, color=C_MUTED,
             ha='left', va='top', alpha=0.5)

    # Bottom: Difference bars
    colors_bar = np.where(diffs > 0, C_RAM, C_LENT)
    colors_bar = np.where(np.abs(diffs) < 7, C_ACCENT, colors_bar)
    ax2.bar(years, diffs, color=colors_bar, width=1.0, alpha=0.8)
    ax2.axhline(0, color=C_MUTED, linewidth=0.7)
    ax2.axhspan(-7, 7, alpha=0.06, color=C_ACCENT)
    ax2.text(years.max() + 3, 0, '±7 dni', fontsize=8, color=C_ACCENT, va='center')
    ax2.set_xlabel('Rok', fontsize=11)
    ax2.set_ylabel('Δ dni', fontsize=10)
    ax2.set_title('Różnica: Ramadan − Środa Popielcowa (dni)',
                  fontsize=10, color=C_MUTED)
    ax2.grid(True, axis='y', linewidth=0.4)

    for ax in axes:
        ax.set_xlim(START_YEAR - 2, END_YEAR + 2)

    plt.tight_layout(pad=2.0)
    path = f'{OUT}/01_hero_timeline.png'
    plt.savefig(path, dpi=180, bbox_inches='tight', facecolor=C_BG)
    plt.close()
    print(f'✓ {path}')

chart_hero()


# ══════════════════════════════════════════════════════════════════════
#  CHART 2 — RAMADAN DRIFT (scatter)
# ══════════════════════════════════════════════════════════════════════

def chart_ramadan_drift():
    fig, ax = plt.subplots(figsize=(16, 9), facecolor=C_BG)

    sc = ax.scatter(years, rd_doy, c=years, cmap='plasma', s=10, alpha=0.85, zorder=3)
    cbar = plt.colorbar(sc, ax=ax, pad=0.02, aspect=40)
    cbar.set_label('Rok', fontsize=11, color=C_TEXT)
    cbar.ax.yaxis.set_tick_params(color=C_TEXT)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=C_TEXT)

    # Ash Wednesday band
    aw_min, aw_max = aw_doy.min(), aw_doy.max()
    ax.axhspan(aw_min, aw_max, alpha=0.12, color=C_LENT, zorder=1)
    ax.text(START_YEAR + 8, (aw_min + aw_max) / 2,
            'Zakres Środy\nPopielcowej', fontsize=9, color=C_LENT,
            va='center', fontweight='bold', alpha=0.8)

    ax.set_yticks(MONTH_DOY)
    ax.set_yticklabels(MONTH_PL, fontsize=10)
    ax.set_ylim(1, 366)
    ax.set_xlim(START_YEAR - 2, END_YEAR + 2)

    ax.set_title('Dryfowanie Ramadanu przez kalendarz gregoriański (1800–2099)',
                 fontsize=15, fontweight='bold', color='white', pad=12)
    ax.set_xlabel('Rok', fontsize=11)
    ax.set_ylabel('Dzień w roku (1 Ramadan)', fontsize=11)
    ax.grid(True, alpha=0.25, linewidth=0.4)

    # Cycle annotation
    n_cycles = (END_YEAR - START_YEAR) / 33
    ax.text(0.98, 0.02,
            f'Cykl islamski ≈ 354.37 dni → przesunięcie ~10–11 dni/rok\n'
            f'Pełny obrót: ~33 lata | w zakresie {START_YEAR}–{END_YEAR}: ~{n_cycles:.0f} cykli',
            transform=ax.transAxes, fontsize=9, color=C_MUTED,
            ha='right', va='bottom',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=C_CARD, edgecolor=C_GRID))

    plt.tight_layout()
    path = f'{OUT}/02_ramadan_drift.png'
    plt.savefig(path, dpi=180, bbox_inches='tight', facecolor=C_BG)
    plt.close()
    print(f'✓ {path}')

chart_ramadan_drift()


# ══════════════════════════════════════════════════════════════════════
#  CHART 3 — HEATMAP: Decade × Month
# ══════════════════════════════════════════════════════════════════════

def chart_heatmap_decade():
    decades = sorted(set(y // 10 * 10 for y in years))
    month_counts = np.zeros((len(decades), 12))
    for r in rows:
        dec_idx = decades.index(r[0] // 10 * 10)
        month_counts[dec_idx, r[2].month - 1] += 1

    fig, ax = plt.subplots(figsize=(14, 14), facecolor=C_BG)
    im = ax.imshow(month_counts, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    ax.set_xticks(range(12))
    ax.set_xticklabels(MONTH_PL, fontsize=10)
    ax.set_yticks(range(len(decades)))
    ax.set_yticklabels([f'{d}s' for d in decades], fontsize=8)

    for i in range(len(decades)):
        for j in range(12):
            v = int(month_counts[i, j])
            if v > 0:
                ax.text(j, i, str(v), ha='center', va='center',
                        fontsize=7, fontweight='bold',
                        color='white' if v >= 4 else '#333')

    cbar = plt.colorbar(im, ax=ax, pad=0.02, aspect=40)
    cbar.set_label('Liczba lat', fontsize=10, color=C_TEXT)
    cbar.ax.yaxis.set_tick_params(color=C_TEXT)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=C_TEXT)

    ax.set_title('Ramadan wg dekad i miesięcy (1800–2099)',
                 fontsize=14, fontweight='bold', color='white', pad=12)
    ax.set_xlabel('Miesiąc gregoriański', fontsize=11)
    ax.set_ylabel('Dekada', fontsize=11)
    plt.tight_layout()
    path = f'{OUT}/03_heatmap_decades.png'
    plt.savefig(path, dpi=180, bbox_inches='tight', facecolor=C_BG)
    plt.close()
    print(f'✓ {path}')

chart_heatmap_decade()


# ══════════════════════════════════════════════════════════════════════
#  CHART 4 — HISTOGRAM of Δ days
# ══════════════════════════════════════════════════════════════════════

def chart_histogram():
    fig, ax = plt.subplots(figsize=(14, 6), facecolor=C_BG)
    n_vals, bins, patches = ax.hist(diffs, bins=80, color=C_RAM, alpha=0.7,
                                    edgecolor='none')
    for p, left, right in zip(patches, bins[:-1], bins[1:]):
        if abs((left + right) / 2) < 7:
            p.set_facecolor(C_ACCENT)
            p.set_alpha(0.9)

    ax.axvline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)
    ax.axvline(diffs.mean(), color=C_LENT, linewidth=1.5, alpha=0.8)
    ax.text(diffs.mean() + 4, n_vals.max() * 0.92,
            f'Średnia: {diffs.mean():.1f} dni', fontsize=10, color=C_LENT, fontweight='bold')
    ax.axvspan(-7, 7, alpha=0.07, color=C_ACCENT)

    close_pct = np.sum(abs_diffs <= 7) / len(diffs) * 100
    ax.text(0, n_vals.max() * 0.65, f'|Δ| ≤ 7 dni:\n{close_pct:.1f}%',
            fontsize=10, color=C_ACCENT, ha='center', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=C_CARD, edgecolor=C_ACCENT, alpha=0.8))

    ax.set_title('Rozkład różnic: Ramadan − Środa Popielcowa (1500–2099)',
                 fontsize=15, fontweight='bold', color='white', pad=12)
    ax.set_xlabel('Δ dni', fontsize=11)
    ax.set_ylabel('Liczba lat', fontsize=11)
    ax.grid(True, axis='y', linewidth=0.4)
    plt.tight_layout()
    path = f'{OUT}/04_histogram_diff.png'
    plt.savefig(path, dpi=180, bbox_inches='tight', facecolor=C_BG)
    plt.close()
    print(f'✓ {path}')

chart_histogram()


# ══════════════════════════════════════════════════════════════════════
#  CHART 5 — PROXIMITY WINDOWS (|Δ| bars, colored by proximity)
# ══════════════════════════════════════════════════════════════════════

def chart_proximity_windows():
    fig, ax = plt.subplots(figsize=(20, 5.5), facecolor=C_BG)

    ax.bar(years, abs_diffs, width=1.0, color=C_MUTED, alpha=0.2)
    mask_14 = abs_diffs <= 14
    mask_7  = abs_diffs <= 7
    mask_0  = abs_diffs == 0
    ax.bar(years[mask_14], abs_diffs[mask_14], width=1.0, color=C_RAM, alpha=0.5)
    ax.bar(years[mask_7], abs_diffs[mask_7], width=1.0, color=C_ACCENT, alpha=0.7)
    ax.bar(years[mask_0], abs_diffs[mask_0] + 2, width=1.5, color='#ef4444', alpha=0.9)

    ax.axhline(14, color=C_RAM, linewidth=0.7, linestyle='--', alpha=0.4)
    ax.axhline(7, color=C_ACCENT, linewidth=0.7, linestyle='--', alpha=0.4)

    patches_legend = [
        mpatches.Patch(color='#ef4444', alpha=0.9, label='Ten sam dzień'),
        mpatches.Patch(color=C_ACCENT, alpha=0.7, label='≤ 7 dni'),
        mpatches.Patch(color=C_RAM, alpha=0.5, label='≤ 14 dni'),
        mpatches.Patch(color=C_MUTED, alpha=0.2, label='> 14 dni'),
    ]
    ax.legend(handles=patches_legend, fontsize=9, loc='upper right',
              facecolor=C_CARD, edgecolor=C_GRID, labelcolor=C_TEXT)

    ax.set_title('Okna bliskości — odległość dat startowych obu postów (1800–2099)',
                 fontsize=14, fontweight='bold', color='white', pad=12)
    ax.set_xlabel('Rok', fontsize=11)
    ax.set_ylabel('|Δ| dni', fontsize=10)
    ax.set_xlim(START_YEAR - 2, END_YEAR + 2)
    ax.grid(True, axis='y', linewidth=0.4)
    plt.tight_layout()
    path = f'{OUT}/05_proximity_windows.png'
    plt.savefig(path, dpi=180, bbox_inches='tight', facecolor=C_BG)
    plt.close()
    print(f'✓ {path}')

chart_proximity_windows()


# ══════════════════════════════════════════════════════════════════════
#  CHART 6 — POLAR: Ramadan month distribution
# ══════════════════════════════════════════════════════════════════════

def chart_polar():
    month_count = np.zeros(12)
    for r in rows:
        month_count[r[2].month - 1] += 1

    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'projection': 'polar'},
                           facecolor=C_BG)
    ax.set_facecolor(C_BG)

    angles = np.linspace(0, 2 * np.pi, 12, endpoint=False)
    widths = 2 * np.pi / 12
    bars = ax.bar(angles, month_count, width=widths * 0.85, alpha=0.8,
                  color=C_RAM, edgecolor=C_BG, linewidth=1.5)

    aw_months = set(r[1].month - 1 for r in rows)
    for idx in aw_months:
        bars[idx].set_facecolor(C_LENT)
        bars[idx].set_alpha(0.6)

    ax.set_xticks(angles)
    ax.set_xticklabels(MONTH_PL, fontsize=11, color=C_TEXT)
    ax.set_title(f'Rozkład 1. Ramadanu\nw miesiącach gregoriańskich\n({START_YEAR}–{END_YEAR})',
                 fontsize=13, fontweight='bold', color='white', pad=25)
    ax.yaxis.set_tick_params(labelcolor=C_MUTED, labelsize=8)
    ax.spines['polar'].set_color(C_GRID)

    patches_legend = [
        mpatches.Patch(color=C_RAM, alpha=0.8, label='Ramadan'),
        mpatches.Patch(color=C_LENT, alpha=0.6, label='Miesiące pokrywające się ze Śr. Pop.'),
    ]
    ax.legend(handles=patches_legend, fontsize=9, loc='lower right',
              bbox_to_anchor=(1.25, -0.05),
              facecolor=C_CARD, edgecolor=C_GRID, labelcolor=C_TEXT)
    plt.tight_layout()
    path = f'{OUT}/06_polar_months.png'
    plt.savefig(path, dpi=180, bbox_inches='tight', facecolor=C_BG)
    plt.close()
    print(f'✓ {path}')

chart_polar()


# ══════════════════════════════════════════════════════════════════════
#  CHART 7 — ROLLING PROXIMITY INDEX (33-year window)
# ══════════════════════════════════════════════════════════════════════

def chart_rolling_proximity():
    window = 33
    abs_d = abs_diffs.astype(float)
    kernel = np.ones(window) / window
    rolling = np.convolve(abs_d, kernel, mode='valid')
    x_roll = years[window - 1:][:len(rolling)]

    fig, ax = plt.subplots(figsize=(18, 5.5), facecolor=C_BG)
    ax.fill_between(x_roll, rolling, alpha=0.15, color=C_RAM)
    ax.plot(x_roll, rolling, color=C_RAM, linewidth=2, label=f'Śr. krocząca ({window} lat)')

    # Mark all local minima
    from scipy.signal import argrelextrema
    try:
        local_min_idx = argrelextrema(rolling, np.less, order=12)[0]
    except:
        local_min_idx = [np.argmin(rolling)]

    for idx in local_min_idx:
        ax.plot(x_roll[idx], rolling[idx], 'o', color=C_ACCENT, markersize=8,
                markeredgecolor='white', markeredgewidth=1.2, zorder=5)
        ax.annotate(f'{x_roll[idx]}',
                    (x_roll[idx], rolling[idx]),
                    fontsize=7, fontweight='bold', color=C_ACCENT,
                    textcoords='offset points', xytext=(0, -14), ha='center')

    ax.set_title(f'Indeks bliskości — średnia krocząca |Δ| ({window}-letnie okno ≈ 1 cykl hidżry)',
                 fontsize=14, fontweight='bold', color='white', pad=12)
    ax.set_xlabel('Rok', fontsize=11)
    ax.set_ylabel(f'Średnia |Δ| (dni)', fontsize=10)
    ax.legend(fontsize=10, facecolor=C_CARD, edgecolor=C_GRID, labelcolor=C_TEXT)
    ax.grid(True, linewidth=0.4)
    ax.set_xlim(START_YEAR - 2, END_YEAR + 2)
    plt.tight_layout()
    path = f'{OUT}/07_rolling_proximity.png'
    plt.savefig(path, dpi=180, bbox_inches='tight', facecolor=C_BG)
    plt.close()
    print(f'✓ {path}')

chart_rolling_proximity()


# ══════════════════════════════════════════════════════════════════════
#  CHART 8 — OVERLAP GANTT (2020–2035)
# ══════════════════════════════════════════════════════════════════════

def chart_overlap_gantt():
    LENT_DAYS = 46
    RAMADAN_DAYS = 30
    recent = [(y, aw, rd, d, s) for (y, aw, rd, d, s) in rows if 2020 <= y <= 2035]
    if not recent:
        return

    fig, ax = plt.subplots(figsize=(16, 8), facecolor=C_BG)
    bar_h = 0.35

    for i, (yr, aw, rd, diff, same) in enumerate(recent):
        aw_doy_val = aw.timetuple().tm_yday
        rd_doy_val = rd.timetuple().tm_yday
        ax.barh(i + bar_h / 2, LENT_DAYS, left=aw_doy_val, height=bar_h,
                color=C_LENT, alpha=0.85)
        ax.barh(i - bar_h / 2, RAMADAN_DAYS, left=rd_doy_val, height=bar_h,
                color=C_RAM, alpha=0.85)
        # Overlap
        lent_end = aw_doy_val + LENT_DAYS
        ram_end = rd_doy_val + RAMADAN_DAYS
        ov_start = max(aw_doy_val, rd_doy_val)
        ov_end = min(lent_end, ram_end)
        if ov_end > ov_start:
            ov_days = ov_end - ov_start
            ax.barh(i, ov_days, left=ov_start, height=bar_h * 2.2,
                    color=C_ACCENT, alpha=0.3, edgecolor=C_ACCENT, linewidth=1)
            ax.text(ov_start + ov_days / 2, i, f'{ov_days}d',
                    fontsize=8, color=C_ACCENT, ha='center', va='center', fontweight='bold')

    ax.set_yticks(range(len(recent)))
    ax.set_yticklabels([str(r[0]) for r in recent], fontsize=10)
    ax.set_xticks(MONTH_DOY[:9])
    ax.set_xticklabels(MONTH_PL[:9], fontsize=10)
    ax.set_xlim(15, 260)

    patches_legend = [
        mpatches.Patch(color=C_LENT, alpha=0.85, label='Wielki Post (46 dni)'),
        mpatches.Patch(color=C_RAM, alpha=0.85, label='Ramadan (~30 dni)'),
        mpatches.Patch(color=C_ACCENT, alpha=0.3, label='Nakładanie się'),
    ]
    ax.legend(handles=patches_legend, fontsize=10, loc='lower right',
              facecolor=C_CARD, edgecolor=C_GRID, labelcolor=C_TEXT)
    ax.set_title('Nakładanie się okresów Wielkiego Postu i Ramadanu (2020–2035)',
                 fontsize=14, fontweight='bold', color='white', pad=12)
    ax.set_xlabel('Dzień roku', fontsize=11)
    ax.grid(True, axis='x', linewidth=0.4)
    ax.invert_yaxis()
    plt.tight_layout()
    path = f'{OUT}/08_overlap_gantt.png'
    plt.savefig(path, dpi=180, bbox_inches='tight', facecolor=C_BG)
    plt.close()
    print(f'✓ {path}')

chart_overlap_gantt()


# ══════════════════════════════════════════════════════════════════════
#  CHART 9 — CONVERGENCE CYCLE: ~33-year periodicity
# ══════════════════════════════════════════════════════════════════════

def chart_convergence_cycle():
    """Fold data by 33-year cycle to show periodic structure."""
    cycle = 33
    cycle_pos = (years - years.min()) % cycle

    fig, ax = plt.subplots(figsize=(12, 7), facecolor=C_BG)

    sc = ax.scatter(cycle_pos, abs_diffs, c=years, cmap='viridis',
                    s=15, alpha=0.6, zorder=3)
    cbar = plt.colorbar(sc, ax=ax, pad=0.02, aspect=40)
    cbar.set_label('Rok', fontsize=10, color=C_TEXT)
    cbar.ax.yaxis.set_tick_params(color=C_TEXT)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=C_TEXT)

    # Bin averages
    bin_means = []
    for b in range(cycle):
        mask = cycle_pos == b
        if mask.any():
            bin_means.append((b, abs_diffs[mask].mean()))
    bx, by = zip(*bin_means)
    ax.plot(bx, by, 'o-', color=C_ACCENT, linewidth=2, markersize=6,
            label='Średnia |Δ| w fazie cyklu', zorder=5)

    ax.set_title('Faza 33-letniego cyklu a odległość dat postów',
                 fontsize=14, fontweight='bold', color='white', pad=12)
    ax.set_xlabel('Pozycja w 33-letnim cyklu (lata)', fontsize=11)
    ax.set_ylabel('|Δ| dni', fontsize=11)
    ax.legend(fontsize=10, facecolor=C_CARD, edgecolor=C_GRID, labelcolor=C_TEXT)
    ax.grid(True, linewidth=0.4)
    ax.set_xlim(-0.5, cycle - 0.5)

    plt.tight_layout()
    path = f'{OUT}/09_convergence_cycle.png'
    plt.savefig(path, dpi=180, bbox_inches='tight', facecolor=C_BG)
    plt.close()
    print(f'✓ {path}')

chart_convergence_cycle()


# ══════════════════════════════════════════════════════════════════════
#  CHART 10 — FUTURE CONVERGENCES (next 75 years)
# ══════════════════════════════════════════════════════════════════════

def chart_future():
    """Horizontal timeline of upcoming convergences."""
    future = [(y, aw, rd, d, s) for (y, aw, rd, d, s) in rows
              if 2025 <= y <= 2099 and abs(d) <= 30]

    if not future:
        print('⚠ No future convergences found')
        return

    fig, ax = plt.subplots(figsize=(18, 6), facecolor=C_BG)

    for i, (yr, aw, rd, diff, same) in enumerate(future):
        color = '#ef4444' if same else (C_ACCENT if abs(diff) <= 7 else C_RAM)
        alpha = 1.0 if same else (0.8 if abs(diff) <= 7 else 0.5)
        size = 250 if same else (150 if abs(diff) <= 7 else 80)

        ax.scatter(yr, abs(diff), c=color, s=size, alpha=alpha, zorder=5,
                   edgecolors='white', linewidths=0.8)
        ax.annotate(f'{yr}\n({diff:+d}d)',
                    (yr, abs(diff)),
                    fontsize=7, color=color, ha='center',
                    textcoords='offset points', xytext=(0, 12))

    ax.axhline(0, color=C_MUTED, linewidth=0.5)
    ax.axhline(7, color=C_ACCENT, linewidth=0.7, linestyle='--', alpha=0.4)
    ax.axhline(14, color=C_RAM, linewidth=0.7, linestyle='--', alpha=0.3)

    ax.set_title('Najbliższe zbieżności: Wielki Post ↔ Ramadan (|Δ| ≤ 30 dni, 2025–2099)',
                 fontsize=14, fontweight='bold', color='white', pad=12)
    ax.set_xlabel('Rok', fontsize=11)
    ax.set_ylabel('|Δ| dni', fontsize=10)
    ax.set_xlim(2023, 2101)
    ax.grid(True, linewidth=0.4)
    plt.tight_layout()
    path = f'{OUT}/10_future_convergences.png'
    plt.savefig(path, dpi=180, bbox_inches='tight', facecolor=C_BG)
    plt.close()
    print(f'✓ {path}')

chart_future()


# ══════════════════════════════════════════════════════════════════════
#  SUMMARY & INTERPRETIVE TEXT
# ══════════════════════════════════════════════════════════════════════

close_7  = np.sum(abs_diffs <= 7)
close_14 = np.sum(abs_diffs <= 14)
close_30 = np.sum(abs_diffs <= 30)

summary = f"""
═══════════════════════════════════════════════════════════════════════
  STATYSTYKI ({START_YEAR}–{END_YEAR}, n={len(rows)} par dat)
═══════════════════════════════════════════════════════════════════════

  Średnia |Δ|:              {abs_diffs.mean():.1f} dni
  Mediana |Δ|:              {np.median(abs_diffs):.0f} dni
  Min / Max |Δ|:            {abs_diffs.min()} / {abs_diffs.max()} dni
  Odch. std. Δ:             {diffs.std():.1f} dni

  Ten sam dzień (Δ=0):      {sames.sum()} razy → {', '.join(map(str, same_years))}
  |Δ| ≤  7 dni:             {close_7} ({close_7/len(rows)*100:.1f}%)
  |Δ| ≤ 14 dni:             {close_14} ({close_14/len(rows)*100:.1f}%)
  |Δ| ≤ 30 dni:             {close_30} ({close_30/len(rows)*100:.1f}%)

  Źródło dat Ramadanu:
    Umm al-Qura:             {sum(is_uaq)} lat ({UQ_MIN_Y}–{UQ_MAX_Y})
    Aproksymacja liniowa:    {sum(~is_uaq)} lat (±1–2 dni błędu)

═══════════════════════════════════════════════════════════════════════

OPISY INTERPRETACYJNE — DO ARTYKUŁU / POSTA
═══════════════════════════════════════════════════════════════════════

📌 KONTEKST — DWA KALENDARZE, DWA SYSTEMY POSTNE

Wielki Post i Ramadan to najważniejsze okresy postu w dwóch
spośród trzech największych religii świata. Rządzą się jednak
zupełnie innymi regułami kalendarzowymi:

• Środa Popielcowa jest zaklinowana w wąskim paśmie luty–marzec
  (4 II – 10 III), bo Wielkanoc zależy od pełni Księżyca po
  równonocy wiosennej (22 III – 25 IV). Zakres: ~35 dni.

• 1 Ramadan wędruje po CAŁYM kalendarzu gregoriańskim, przesuwając
  się o ~10–11 dni rocznie (rok islamski = ~354.37 dni). Pełny
  obrót co ~33 lata.

📌 ZBIEŻNOŚĆ TO RZADKOŚĆ

Z {len(rows)} zbadanych par dat (600 lat!) wynika:
• Identyczna data startu (Δ=0) zdarzyła się tylko {sames.sum()} razy.
  Lata: {', '.join(map(str, same_years))}.
• Odstęp ≤ 7 dni: {close_7} razy ({close_7/len(rows)*100:.1f}%)
  → średnio raz na ~{600//max(close_7,1)} lat.
• Średnia odległość to {abs_diffs.mean():.0f} dni — potwierdzenie,
  że w typowym roku daty nie mają ze sobą „nic wspólnego".

📌 CYKLICZNOŚĆ

Wykres rolling proximity (33-letnie okno) i wykres faz cyklu
ujawniają deterministyczną falistość: co ~33 lata następuje
„okno zbieżności" trwające kilka lat, po czym odległość rośnie
do maksimum i znów maleje.

📌 NAKŁADANIE SIĘ OKRESÓW

Nawet gdy daty startowe się nie pokrywają, OKRESY postów mogą
się nakładać (Wielki Post: 46 dni, Ramadan: ~30 dni). Wykres
Gantta (chart 8) pokazuje np. 2026: 30 dni wspólnego postu.

📌 ROK 2026 W KONTEKŚCIE HISTORYCZNYM

2026 to pierwszy rok od 1928, w którym oba posty zaczynają się
tego samego dnia. Następna taka zbieżność (Δ=0) to dopiero
{next((y for y in same_years if y > 2026), '?')}.
To czyni 2026 wyjątkowym momentem dla dialogu międzyreligijnego.

📌 ZASTRZEŻENIA

• Daty Ramadanu sprzed 1924 i po 2077 to aproksymacja liniowa
  (błąd ≤ ±2 dni vs Umm al-Qura). Dla wizualizacji trendów
  wieloletnich to wystarczająca dokładność.
• W praktyce wiele społeczności muzułmańskich rozpoczyna Ramadan
  na podstawie lokalnej obserwacji nowiu — przesunięcie ±1–2 dni.
• Środa Popielcowa wg kalendarza gregoriańskiego (zachodniego).
  Prawosławna data może różnić się o 1–5 tygodni.
═══════════════════════════════════════════════════════════════════════
"""

print(summary)

with open(f'{OUT}/opisy_interpretacyjne.txt', 'w', encoding='utf-8') as f:
    f.write(summary)
print(f'✓ Tekst zapisany: {OUT}/opisy_interpretacyjne.txt')

Wygenerowano 414 par dat (1800–2200)
Pokrycia (Δ=0): [np.int64(1830), np.int64(1928), np.int64(2026), np.int64(2124), np.int64(2155)]
✓ /home/claude/output/01_hero_timeline.png
✓ /home/claude/output/02_ramadan_drift.png
✓ /home/claude/output/03_heatmap_decades.png
✓ /home/claude/output/04_histogram_diff.png
✓ /home/claude/output/05_proximity_windows.png
✓ /home/claude/output/06_polar_months.png
✓ /home/claude/output/07_rolling_proximity.png
✓ /home/claude/output/08_overlap_gantt.png
✓ /home/claude/output/09_convergence_cycle.png
✓ /home/claude/output/10_future_convergences.png

═══════════════════════════════════════════════════════════════════════
  STATYSTYKI (1800–2200, n=414 par dat)
═══════════════════════════════════════════════════════════════════════

  Średnia |Δ|:              139.2 dni
  Mediana |Δ|:              132 dni
  Min / Max |Δ|:            0 / 327 dni
  Odch. std. Δ:             107.2 dni

  Ten sam dzień (Δ=0):      5 razy → 1830, 1928, 2026, 2124, 2155
  |Δ| ≤  7 